# Chapitre 7 - Cours 3 : mises à jour perdues, write skew et fantômes

Ce notebook couvre la fin de la section *Weak Isolation Levels* : *Preventing Lost
Updates* (*Atomic write operations*, *Explicit locking*, *Automatically detecting lost
updates*, *Compare-and-set*, *Conflict resolution and
replication*) et *Write Skew and Phantoms* (*Characterizing write skew*, *More examples of
write skew*, *Phantoms causing write skew*, *Materializing conflicts*). Le cours 2 a
traité ce que voit un **lecteur** ; ici, deux transactions **écrivent** en même temps.

**Plan.**

1. La mise à jour perdue : le cycle lire-modifier-écrire.
2. Opérations d'écriture atomiques.
3. Verrouillage explicite (`SELECT ... FOR UPDATE`) et interblocages.
4. Détection automatique des mises à jour perdues.
5. Écriture conditionnelle (*compare-and-set*).
6. Réplication : résolution des conflits.
7. Le write skew : les médecins de garde.
8. Autres exemples ; les fantômes.
9. Matérialiser les conflits.
10. Récapitulatif expérimental : quel niveau empêche quoi ?

In [1]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import sqlite3

from ddia import transactions as tr
from ddia.transactions import (R, W, S, INC, RFU, SFU, run, plot_history, plot_precedence,
                               precedence_graph, simulate_load, LEVEL_LABELS)
from ddia.diagrams import Diagram
from ddia.plotting import (AMBER, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET,
                           annotate_bars, finish, set_style, subtitle)

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 3)
rng = np.random.default_rng(0)